## Settlements → Radio Station Coverage
Spatial join of surveyed settlements (points) to CloudRF radio station reach polygons (`.gpkg`).
Loading pattern follows `pipeline_output/pipeline/etl_radio.py :: add_station_labels`.

Results of this script are used to inform the phone pulse survey radio select options, based on the respondent's location.

In [ ]:
import re
import glob
import pandas as pd
import geopandas as gpd
from src.config import DIR_META,DIR_WEIGHTED_DATA, DIR_META_DATA, DIR_STATION

MAX_SNAP_DISTANCE = 2000  # metres — points beyond this stay unmatched

In [ ]:
survey = pd.read_csv(DIR_WEIGHTED_DATA)
survey = gpd.GeoDataFrame(survey, geometry=gpd.points_from_xy(survey['gps-Longitude'], survey['gps-Latitude']), crs="EPSG:4326")


In [ ]:
survey.columns

In [ ]:
# Filter only consenting respondents with valid phone numbers (at least 8 digits)
phone_survey = survey.loc[(survey.phonenumber_futurecontact=="Oui / Ɛɛn") & (survey.phonenumber_number.map(lambda x: len(str(x)) >= 8))].copy()
len(phone_survey)

In [ ]:
phone_survey.groupby('gender').count()

Join to radio rasters for analysis:

In [ ]:
# Select metric CRS suitable for country
crs_metric = "epsg:32632"

In [ ]:
# Load all station polygons — same glob pattern as etl_radio.py
station_files = glob.glob(DIR_STATION + "*.gpkg")
print(f"{len(station_files)} station files found")

# Reproject
survey = survey.to_crs(crs_metric)

# Initialise lists
all_stations = []
all_station_names = []  # all 24 station names regardless of hit count
all_hits = []           # long-format: one entry per (respondent, station) pair

BUFFER_M = 1000  # metres added to each station polygon before spatial join
survey['geometry'] = survey.geometry.buffer(BUFFER_M)

# Iterate through station files and assess how many fall within radio boundaries.
for station_file in station_files:
    station_gdf = gpd.read_file(station_file)
    if station_gdf.crs != survey.crs:
        station_gdf = station_gdf.to_crs(survey.crs)
    station_proj = station_gdf.to_crs(crs_metric)
    station_name = station_file.split("/")[-1].replace(".gpkg", "")
    station_proj["station_name"] = station_name

    all_stations.append(station_proj)
    all_station_names.append(station_name)

    hits = gpd.sjoin(survey, station_proj, how="inner", predicate="intersects")
    for idx in hits.index:
        all_hits.append({"survey_index": idx, "station_label": station_name})
    print(f"  {station_name}: {len(hits)} respondents within bounds")

matched_indices = {h["survey_index"] for h in all_hits}
print(f"\n{len(matched_indices)} unique respondents matched to at least one station")
print(f"{len(all_station_names) - len({h['station_label'] for h in all_hits})} stations with 0 respondents within bounds")

In [ ]:
# Snap unmatched respondents to nearest station (within MAX_SNAP_DISTANCE)
unmatched_idx = [i for i in mapped_data_proj.index if i not in matched_indices]
print(f"{len(unmatched_idx)} respondents unmatched after within-join — snapping...")

if len(unmatched_idx) > 0 and len(all_stations) > 0:
    all_stations_gdf = pd.concat(all_stations, ignore_index=True)
    snapped = 0
    for idx in unmatched_idx:
        point = mapped_data_proj.loc[idx, "geometry"]
        all_stations_gdf["distance"] = all_stations_gdf.geometry.distance(point)
        nearest = all_stations_gdf.loc[all_stations_gdf["distance"].idxmin()]
        if nearest["distance"] <= MAX_SNAP_DISTANCE:
            all_hits.append({"survey_index": idx, "station_label": nearest["station_name"]})
            snapped += 1
    print(f"  Snapped {snapped}; {len(unmatched_idx) - snapped} remain beyond {MAX_SNAP_DISTANCE}m threshold")

In [ ]:
hits_df = pd.DataFrame(all_hits)
result_base = mapped_data_proj.drop(columns="geometry").copy()

# Long-format: one row per (respondent × station)
result = result_base.merge(hits_df, left_index=True, right_on="survey_index", how="left").drop(columns="survey_index")

matched_respondents = hits_df["survey_index"].nunique()
print(f"\nCoverage summary:")
print(f"  Unique respondents matched: {matched_respondents} / {len(mapped_data_proj)}")
print(f"  Unmatched:                  {len(mapped_data_proj) - matched_respondents}")
print(f"  Total (respondent × station) rows: {result['station_label'].notna().sum()}")
print()

# reindex ensures all 24 stations appear even if 0 respondents were within bounds
result.groupby("station_label")["location"].apply(lambda x: sorted(set(x.dropna()))).reindex(all_station_names, fill_value=[])

In [ ]:
result.groupby("station_label")["location"].apply(lambda x: sorted(set(x.dropna())))\
    .reindex(all_station_names, fill_value=[])\
    .to_csv('data/4_validation_checks/stations_with_settlements.csv', encoding='utf-8-sig')


In [ ]:
result.groupby(['location',  "station_label"]).count()['SubmissionDate']\
    .reset_index()\
        .rename(columns = {'SubmissionDate': 'num_respondents'})\
           .to_csv('data/4_validation_checks/settlements_with_stations.csv', encoding='utf-8-sig')   
    
